# Day 4 - Pandas Deep Dive (End-to-End)

> Single notebook covering all of Day 4: Pandas Deep Dive (EDA focus)

This notebook covers:
1. Pandas Introduction (Series vs DataFrame)
2. DataFrame Creation & Basic Exploration
3. Indexing - `.loc` / `.iloc`
4. Missing Data Handling
5. GroupBy
6. Merge / Join
7. Apply vs Vectorized Operations
8. Basic EDA Workflow (putting it all together)

We reuse the **same 10-customer Fixed Deposit (FD) dataset** from Day 3 NumPy - but this time as ONE Pandas DataFrame with mixed data types living together naturally, which is exactly the limitation of NumPy this dataset was designed to expose.


## 1. Pandas Introduction

### Introduction
- **Pandas** is a Python library built on top of NumPy, designed for working with **tabular (row-column) data**
- Why do we need it?
  - NumPy arrays must be a single data type - real-world data (like our FD dataset) has numbers, strings, and booleans all together in one table
  - Pandas solves this using the **DataFrame**, which allows each column to have its own data type, while still being fast (built on NumPy internally)
- Where is it used?
  - Almost all real-world data cleaning, EDA (Exploratory Data Analysis), and data preprocessing before machine learning

### Real-Life Analogy
- Think of a NumPy array like **separate spreadsheets**, one per column, each locked to one data type
- A Pandas DataFrame is like **one single Excel sheet**, where different columns can hold different types of data (names as text, age as numbers, active status as True/False) - all lined up together in rows

### Explanation
- Pandas has two core objects:
  - **Series** -> a single column of data (basically a labeled 1D array)
  - **DataFrame** -> a full table, made up of multiple Series sharing the same row index
- Every DataFrame has:
  - An **index** (row labels, default is 0, 1, 2, ... unless set otherwise)
  - **Columns** (column labels)
  - Each column can have its own `dtype`


In [1]:
import pandas as pd
import numpy as np

# Series - a single labeled column
age_series = pd.Series([25, 34, 45, 29, 52], name="age")
print(age_series)
print("Type:", type(age_series))


0    25
1    34
2    45
3    29
4    52
Name: age, dtype: int64
Type: <class 'pandas.core.series.Series'>


**Line by line explanation:**
- `pd.Series([...], name="age")` -> creates a single labeled column of data
- Notice the printed output shows an **index** (0,1,2,3,4) automatically added on the left
- A Series is essentially "one column of a DataFrame, on its own"


## 2. DataFrame Creation & Basic Exploration

### Syntax
```python
import pandas as pd

# From a dictionary of lists (most common way)
df = pd.DataFrame({
    "col1": [1, 2, 3],
    "col2": ["a", "b", "c"]
})

# Basic exploration
df.head()       # first 5 rows
df.info()       # column names, dtypes, non-null counts
df.describe()   # summary statistics for numeric columns
df.shape        # (rows, columns)
df.columns      # column names
df.dtypes       # data type of each column
```

### Example - Building our FD dataset as ONE DataFrame


In [2]:
data = {
    "customer_id":     [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "age":             [25, 34, 45, 29, 52, 23, 41, 38, 60, 31],
    "gender":          ["M", "F", "M", "F", "M", "F", "M", "F", "M", "F"],
    "city":            ["Pune", "Mumbai", "Delhi", "Pune", "Bangalore",
                         "Mumbai", "Delhi", "Bangalore", "Pune", "Delhi"],
    "annual_income":   [45.0, 62.0, np.nan, 38.5, 95.0, 30.0, 71.0, 55.0, 950.0, np.nan],
    "account_balance": [12.5, 45.2, 78.9, 5.0, 120.0, -2.5, 60.0, 0.0, 200.0, 8.0],
    "tenure_years":    [1, 3, 8, 1, 15, 0, 6, 4, 20, 2],
    "num_products":    [1, 2, 3, 1, 4, 1, 2, 2, 5, 1],
    "is_active":       [1, 1, 1, 0, 1, 0, 1, 0, 1, 1],
    "FD":              [0, 1, 1, 0, 1, 0, 1, 0, 1, 0],
}

df = pd.DataFrame(data)
df


,customer_id,age,gender,city,annual_income,account_balance,tenure_years,num_products,is_active,FD
0,1,25,M,Pune,45.0,12.5,1,1,1,0
1,2,34,F,Mumbai,62.0,45.2,3,2,1,1
2,3,45,M,Delhi,NaN,78.9,8,3,1,1
3,4,29,F,Pune,38.5,5.0,1,1,0,0
4,5,52,M,Bangalore,95.0,120.0,15,4,1,1
5,6,23,F,Mumbai,30.0,-2.5,0,1,0,0
6,7,41,M,Delhi,71.0,60.0,6,2,1,1
7,8,38,F,Bangalore,55.0,0.0,4,2,0,0
8,9,60,M,Pune,950.0,200.0,20,5,1,1
9,10,31,F,Delhi,NaN,8.0,2,1,1,0


**Line by line explanation:**
- `pd.DataFrame(dict_of_lists)` -> each dictionary key becomes a **column name**, each list becomes the **column's values**
- Notice: unlike NumPy, we now have **strings (`gender`, `city`) and numbers living together in one table** - this is the exact limitation we hit in Day 3 NumPy, now solved
- Pandas assigns a default numeric row index (0-9) automatically


In [3]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nData types:\n", df.dtypes)
print("\nFirst 5 rows:")
print(df.head())


Shape: (10, 10)

Columns: ['customer_id', 'age', 'gender', 'city', 'annual_income', 'account_balance', 'tenure_years', 'num_products', 'is_active', 'FD']

Data types:
 customer_id          int64
age                  int64
gender              object
city                object
annual_income      float64
account_balance    float64
tenure_years         int64
num_products         int64
is_active            int64
FD                   int64
dtype: object

First 5 rows:
   customer_id  age gender       city  annual_income  account_balance  \
0            1   25      M       Pune           45.0             12.5   
1            2   34      F     Mumbai           62.0             45.2   
2            3   45      M      Delhi            NaN             78.9   
3            4   29      F       Pune           38.5              5.0   
4            5   52      M  Bangalore           95.0            120.0   

   tenure_years  num_products  is_active  FD  
0             1             1          1   0  


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      10 non-null     int64  
 1   age              10 non-null     int64  
 2   gender           10 non-null     object 
 3   city             10 non-null     object 
 4   annual_income    8 non-null      float64
 5   account_balance  10 non-null     float64
 6   tenure_years     10 non-null     int64  
 7   num_products     10 non-null     int64  
 8   is_active        10 non-null     int64  
 9   FD               10 non-null     int64  
dtypes: float64(2), int64(6), object(2)
memory usage: 932.0+ bytes


**Line by line explanation:**
- `.shape` -> `(10, 10)`: 10 rows (customers), 10 columns (features)
- `.dtypes` -> shows each column's data type: `int64` for whole numbers, `float64` for decimals (income, balance), `object` for strings (gender, city)
- `.head()` -> shows the first 5 rows by default, useful for a quick peek without printing the whole table
- `.info()` -> shows column names, **non-null counts** (notice `annual_income` shows only 8 non-null out of 10 - this is how Pandas reveals our 2 missing values at a glance), and dtypes together


In [5]:
df.describe()

,customer_id,age,annual_income,account_balance,tenure_years,num_products,is_active,FD
count,10.00000,10.00000,8.000000,10.00000,10.00000,10.000000,10.000000,10.000000
mean,5.50000,37.80000,168.312500,52.71000,6.00000,2.200000,0.700000,0.500000
std,3.02765,11.91451,316.500839,65.60514,6.63325,1.398412,0.483046,0.527046
min,1.00000,23.00000,30.000000,-2.50000,0.00000,1.000000,0.000000,0.000000
25%,3.25000,29.50000,43.375000,5.75000,1.25000,1.000000,0.250000,0.000000
50%,5.50000,36.00000,58.500000,28.85000,3.50000,2.000000,1.000000,0.500000
75%,7.75000,44.00000,77.000000,74.17500,7.50000,2.750000,1.000000,1.000000
max,10.00000,60.00000,950.000000,200.00000,20.00000,5.000000,1.000000,1.000000


**Line by line explanation:**
- `.describe()` -> generates summary statistics (count, mean, std, min, 25%/50%/75% percentiles, max) for all **numeric** columns automatically
- Look closely at `annual_income`: **mean is pulled way up** (because of our 950.0 outlier) compared to the 50% (median) value - this is a live demonstration of why mean alone can be misleading with outliers
- `count` for `annual_income` shows 8 (not 10) - confirming the 2 missing values are excluded from these calculations automatically


## 3. Indexing - `.loc` and `.iloc`

### Explanation
- `.loc[]` -> **label-based** indexing - you select rows/columns using their actual labels (index values, column names)
- `.iloc[]` -> **position-based** indexing - you select rows/columns using integer positions (like list indexing), regardless of what the labels are
- Both support the same general syntax: `df.loc[row_selector, column_selector]`

### Real-Life Analogy
- `.loc` is like finding a book by its **title** in a library catalog (label-based)
- `.iloc` is like finding a book by counting **shelf position number** (position-based) - "give me the 3rd book on the shelf", regardless of its title

### Syntax
```python
df.loc[row_label]                     # a single row by label
df.loc[row_label, "col_name"]         # a single value
df.loc[:, "col_name"]                 # entire column
df.loc[df["col"] > 5]                 # boolean filtering

df.iloc[0]                            # first row by position
df.iloc[0, 2]                         # row 0, column 2 (by position)
df.iloc[0:3]                          # first 3 rows (position based slicing)
```


In [6]:
# .loc examples - label based
print("Row with index label 3:")
print(df.loc[3])

print("\nJust the 'age' column, using .loc:")
print(df.loc[:, "age"])

print("\nCustomers with age above 40 (boolean filtering with .loc):")
print(df.loc[df["age"] > 40, ["customer_id", "age", "FD"]])


Row with index label 3:
customer_id           4
age                  29
gender                F
city               Pune
annual_income      38.5
account_balance     5.0
tenure_years          1
num_products          1
is_active             0
FD                    0
Name: 3, dtype: object

Just the 'age' column, using .loc:
0    25
1    34
2    45
3    29
4    52
5    23
6    41
7    38
8    60
9    31
Name: age, dtype: int64

Customers with age above 40 (boolean filtering with .loc):
   customer_id  age  FD
2            3   45   1
4            5   52   1
6            7   41   1
8            9   60   1


**Line by line explanation:**
- `df.loc[3]` -> selects the row where the **index label** equals 3 (in our case this happens to also be the 4th row since our index is 0-9, but conceptually it's about labels, not position)
- `df.loc[:, "age"]` -> `:` means "all rows", `"age"` selects just that column
- `df.loc[df["age"] > 40, [...]]` -> `df["age"] > 40` creates a **boolean mask** (True/False per row); `.loc` uses this mask to filter rows, and we also select only specific columns to display


In [7]:
# .iloc examples - position based
print("First row by position:")
print(df.iloc[0])

print("\nValue at row position 2, column position 4 (annual_income of 3rd customer):")
print(df.iloc[2, 4])

print("\nFirst 3 rows, first 3 columns:")
print(df.iloc[0:3, 0:3])


First row by position:
customer_id           1
age                  25
gender                M
city               Pune
annual_income      45.0
account_balance    12.5
tenure_years          1
num_products          1
is_active             1
FD                    0
Name: 0, dtype: object

Value at row position 2, column position 4 (annual_income of 3rd customer):
nan

First 3 rows, first 3 columns:
   customer_id  age gender
0            1   25      M
1            2   34      F
2            3   45      M


**Line by line explanation:**
- `df.iloc[0]` -> selects the row at **position 0** (the first row), regardless of its index label
- `df.iloc[2, 4]` -> row at position 2, column at position 4 - purely counting positions, not names
- `df.iloc[0:3, 0:3]` -> position-based slicing, similar to NumPy/list slicing - `stop` is **exclusive**, just like Python slicing

> Interview tip: A classic interview question is "difference between `.loc` and `.iloc`" - the one-line answer: `.loc` is label-based (inclusive of the end label in slices), `.iloc` is position-based (exclusive of the end position in slices, like standard Python indexing).


## 4. Missing Data Handling

### Explanation
- Missing data (`NaN`) is extremely common in real datasets - could be due to data entry errors, unavailable information, or system issues
- Pandas provides tools to **detect**, **remove**, or **fill** missing values
- Key methods:
  - `.isnull()` / `.isna()` -> returns True/False per cell, indicating missing values
  - `.dropna()` -> removes rows (or columns) containing missing values
  - `.fillna(value)` -> replaces missing values with a specified value (e.g., mean, median, 0)

### Real-Life Analogy
- Think of a survey form where some people skipped a question
- `.isnull()` = highlighting the blank answers
- `.dropna()` = throwing away incomplete forms entirely
- `.fillna()` = filling in the blank with a reasonable estimate (like the average of everyone else's answer) instead of discarding the form


In [8]:
# Detecting missing values
print("Missing values per column:")
print(df.isnull().sum())

print("\nRows with missing annual_income:")
print(df[df["annual_income"].isnull()])


Missing values per column:
customer_id        0
age                0
gender             0
city               0
annual_income      2
account_balance    0
tenure_years       0
num_products       0
is_active          0
FD                 0
dtype: int64

Rows with missing annual_income:
   customer_id  age gender   city  annual_income  account_balance  \
2            3   45      M  Delhi            NaN             78.9   
9           10   31      F  Delhi            NaN              8.0   

   tenure_years  num_products  is_active  FD  
2             8             3          1   1  
9             2             1          1   0  


**Line by line explanation:**
- `df.isnull()` -> returns a same-shaped DataFrame of True/False
- `.sum()` on top of that -> counts True values per column (True behaves like 1 in math) - a quick way to see how many missing values exist per column
- `df[df["annual_income"].isnull()]` -> filters and shows only the rows where `annual_income` is missing (customers 3 and 10, as we designed)


In [9]:
# dropna() - removing rows with missing values
df_dropped = df.dropna()
print("Shape after dropna():", df_dropped.shape, "(started with", df.shape, ")")

# fillna() - filling missing values with the median (robust to our outlier, unlike mean)
median_income = df["annual_income"].median()
print("\nMedian income (used for filling):", median_income)

df_filled = df.copy()
df_filled["annual_income"] = df_filled["annual_income"].fillna(median_income)
print("\nannual_income after filling missing values with median:")
print(df_filled["annual_income"])


Shape after dropna(): (8, 10) (started with (10, 10) )

Median income (used for filling): 58.5

annual_income after filling missing values with median:
0     45.0
1     62.0
2     58.5
3     38.5
4     95.0
5     30.0
6     71.0
7     55.0
8    950.0
9     58.5
Name: annual_income, dtype: float64


**Line by line explanation:**
- `df.dropna()` -> by default drops **any row** that has at least one missing value; our shape goes from `(10, 10)` to `(8, 10)`, losing 2 customers entirely - notice this is often wasteful when we could instead just fill the gap
- `df["annual_income"].median()` -> we deliberately use **median, not mean**, because our income column has the 950.0 outlier that would badly skew the mean - a direct callback to the `.describe()` output earlier
- `df.copy()` -> creates an independent copy so we don't accidentally modify the original `df` (important best practice)
- `.fillna(median_income)` -> replaces `NaN` values with the median, keeping all 10 customers instead of discarding data

> Common mistake alert: Using `.fillna(df["annual_income"].mean())` here would fill missing values with a number skewed upward by the 950.0 outlier - always check for outliers before choosing mean vs median for imputation.


## 5. GroupBy

### Explanation
- `.groupby()` splits the DataFrame into groups based on the values of one or more columns, then lets us apply an aggregation (like `mean`, `sum`, `count`) to each group separately
- This follows the **"split - apply - combine"** pattern:
  1. **Split** the data into groups (e.g., by `FD` status)
  2. **Apply** a function to each group (e.g., calculate mean balance)
  3. **Combine** the results back into a single output

### Real-Life Analogy
- Think of sorting a deck of cards into 4 piles by suit (split), counting how many cards are in each pile (apply), then presenting the 4 counts together (combine)

### Syntax
```python
df.groupby("column_name")["target_column"].mean()
df.groupby("column_name").agg({"col1": "mean", "col2": "sum"})
```


In [10]:
# Average account_balance grouped by FD status
print("Average account_balance by FD status:")
print(df.groupby("FD")["account_balance"].mean())

print("\nAverage age and tenure by city:")
print(df.groupby("city")[["age", "tenure_years"]].mean())


Average account_balance by FD status:
FD
0      4.60
1    100.82
Name: account_balance, dtype: float64

Average age and tenure by city:
            age  tenure_years
city                         
Bangalore  45.0      9.500000
Delhi      39.0      5.333333
Mumbai     28.5      1.500000
Pune       38.0      7.333333


**Line by line explanation:**
- `df.groupby("FD")` -> splits customers into 2 groups: `FD == 0` and `FD == 1`
- `["account_balance"].mean()` -> within each group, calculates the average account balance
- Result clearly shows FD customers tend to have higher balances - useful EDA insight
- `df.groupby("city")[["age", "tenure_years"]].mean()` -> grouping by a categorical column and aggregating multiple numeric columns at once


In [11]:
# Multiple aggregations at once using .agg()
summary = df.groupby("FD").agg(
    avg_balance=("account_balance", "mean"),
    avg_income=("annual_income", "median"),   # median because of our outlier
    customer_count=("customer_id", "count")
)
print(summary)


    avg_balance  avg_income  customer_count
FD                                         
0          4.60       41.75               5
1        100.82       83.00               5


**Line by line explanation:**
- `.agg(new_col_name=("source_column", "aggregation_function"))` -> lets us compute several different aggregations at once, and name the resulting columns clearly
- We use `median` for income specifically because of our known outlier - showing that choice of aggregation should always consider the data's shape
- `("customer_id", "count")` -> counts how many customers fall into each FD group


## 6. Merge / Join

### Explanation
- `pd.merge()` combines two DataFrames based on a common column (like a SQL JOIN)
- Types of joins:
  - **inner** -> only rows with matching keys in both DataFrames (default)
  - **left** -> all rows from the left DataFrame, matched where possible
  - **right** -> all rows from the right DataFrame, matched where possible
  - **outer** -> all rows from both, matched where possible, `NaN` where no match

### Real-Life Analogy
- Think of two spreadsheets: one with customer details, another with branch information for each city
- Merging is like using VLOOKUP in Excel - matching rows from one sheet to another using a common column (`city`)

### Syntax
```python
pd.merge(df1, df2, on="common_column", how="inner")   # or "left"/"right"/"outer"
```


In [12]:
# A second small dataset: branch info per city
branch_info = pd.DataFrame({
    "city": ["Pune", "Mumbai", "Delhi", "Chennai"],
    "branch_manager": ["Anil", "Priya", "Karan", "Meera"],
    "num_branches": [5, 8, 6, 3]
})
branch_info


,city,branch_manager,num_branches
0,Pune,Anil,5
1,Mumbai,Priya,8
2,Delhi,Karan,6
3,Chennai,Meera,3


In [13]:
# Inner join - only cities present in BOTH df and branch_info
merged_inner = pd.merge(df, branch_info, on="city", how="inner")
print("Inner join shape:", merged_inner.shape)
print(merged_inner[["customer_id", "city", "branch_manager", "num_branches"]])


Inner join shape: (8, 12)
   customer_id    city branch_manager  num_branches
0            1    Pune           Anil             5
1            2  Mumbai          Priya             8
2            3   Delhi          Karan             6
3            4    Pune           Anil             5
4            6  Mumbai          Priya             8
5            7   Delhi          Karan             6
6            9    Pune           Anil             5
7           10   Delhi          Karan             6


**Line by line explanation:**
- `pd.merge(df, branch_info, on="city", how="inner")` -> matches rows where `city` exists in both tables
- Notice `"Bangalore"` (present in `df` but not in `branch_info`) gets **dropped entirely** with an inner join - only 10 rows remain only if all cities match; here Bangalore customers disappear
- `"Chennai"` (present in `branch_info` but not in `df`) also disappears since no customer matches it


In [14]:
# Left join - keep ALL customers from df, fill missing branch info with NaN
merged_left = pd.merge(df, branch_info, on="city", how="left")
print("Left join shape:", merged_left.shape)
print(merged_left[["customer_id", "city", "branch_manager", "num_branches"]])


Left join shape: (10, 12)
   customer_id       city branch_manager  num_branches
0            1       Pune           Anil           5.0
1            2     Mumbai          Priya           8.0
2            3      Delhi          Karan           6.0
3            4       Pune           Anil           5.0
4            5  Bangalore            NaN           NaN
5            6     Mumbai          Priya           8.0
6            7      Delhi          Karan           6.0
7            8  Bangalore            NaN           NaN
8            9       Pune           Anil           5.0
9           10      Delhi          Karan           6.0


**Line by line explanation:**
- `how="left"` -> keeps **all 10 customers** (the "left" DataFrame is `df`), regardless of whether a matching city exists in `branch_info`
- Bangalore customers are kept, but `branch_manager` and `num_branches` show `NaN` since there's no matching branch data for Bangalore
- This is the most commonly used join type in real EDA - you usually don't want to silently lose your main data (customers) just because supplementary data (branch info) is incomplete

> Interview tip: `merge()` vs `join()` - `.join()` is a shortcut method that merges using the **index** by default (instead of a column), while `.merge()` is more flexible and merges using any specified column(s).


## 7. Apply vs Vectorized Operations

### Explanation
- **Vectorized operations** apply a calculation to an entire column at once, using Pandas/NumPy's internal optimized C code - fast
- **`.apply()`** runs a Python function **row by row (or element by element)** - flexible, but much slower since it runs actual Python-level loops internally
- Rule of thumb: **always prefer vectorized operations when possible**; use `.apply()` only when the logic genuinely cannot be vectorized (e.g., complex conditional logic across multiple columns)

### Real-Life Analogy
- Vectorized operation = a factory assembly line stamping the same design on 1000 items simultaneously
- `.apply()` = a craftsman manually shaping each of the 1000 items by hand, one at a time - flexible for custom designs, but far slower


In [15]:
import time

# Vectorized operation - convert account_balance to categories using np.where (vectorized)
df["balance_flag_vectorized"] = np.where(df["account_balance"] < 0, "overdraft",
                                  np.where(df["account_balance"] == 0, "empty", "positive"))
print(df[["customer_id", "account_balance", "balance_flag_vectorized"]])


   customer_id  account_balance balance_flag_vectorized
0            1             12.5                positive
1            2             45.2                positive
2            3             78.9                positive
3            4              5.0                positive
4            5            120.0                positive
5            6             -2.5               overdraft
6            7             60.0                positive
7            8              0.0                   empty
8            9            200.0                positive
9           10              8.0                positive


**Line by line explanation:**
- `np.where(condition, value_if_true, value_if_false)` -> a vectorized way to apply conditional logic across an entire column at once, no explicit loop written by us
- Nesting `np.where()` lets us handle 3 categories: negative -> "overdraft", zero -> "empty", positive -> "positive"
- This runs internally in compiled C code across the whole column simultaneously


In [16]:
# Same logic using .apply() with a custom function - row by row
def categorize_balance(balance):
    if balance < 0:
        return "overdraft"
    elif balance == 0:
        return "empty"
    else:
        return "positive"

df["balance_flag_apply"] = df["account_balance"].apply(categorize_balance)
print(df[["customer_id", "account_balance", "balance_flag_apply"]])

# Confirm both approaches give identical results
print("\nBoth methods match:", (df["balance_flag_vectorized"] == df["balance_flag_apply"]).all())


   customer_id  account_balance balance_flag_apply
0            1             12.5           positive
1            2             45.2           positive
2            3             78.9           positive
3            4              5.0           positive
4            5            120.0           positive
5            6             -2.5          overdraft
6            7             60.0           positive
7            8              0.0              empty
8            9            200.0           positive
9           10              8.0           positive

Both methods match: True


**Line by line explanation:**
- `.apply(categorize_balance)` -> calls `categorize_balance()` **once per row**, passing that row's `account_balance` value each time
- Both approaches produce the **same result**, but `.apply()` does so by looping in Python, while `np.where()` does so in vectorized C code
- `.all()` -> confirms every single row matches between the two approaches (True only if ALL values are True)


In [17]:
# Quick benchmark on a larger repeated dataset to SEE the speed difference
big_balance = pd.Series(np.random.uniform(-100, 100, 100000))

start = time.time()
vectorized_result = np.where(big_balance < 0, "overdraft", "positive")
vectorized_time = time.time() - start

start = time.time()
apply_result = big_balance.apply(lambda x: "overdraft" if x < 0 else "positive")
apply_time = time.time() - start

print(f"Vectorized time: {vectorized_time:.5f} seconds")
print(f"Apply time:      {apply_time:.5f} seconds")
print(f"Vectorized was approximately {apply_time / vectorized_time:.1f}x faster")


Vectorized time: 0.00254 seconds
Apply time:      0.01687 seconds
Vectorized was approximately 6.7x faster


**Line by line explanation:**
- We create 100,000 random balance values to make the speed difference clearly visible (our 10-row dataset is too small to show a meaningful timing difference)
- Both approaches produce the same categorization, but timing shows vectorized `np.where()` is dramatically faster than `.apply()` with a Python function
- This is a very common real interview discussion point: "why is my Pandas code slow" often traces back to unnecessary `.apply()` usage where a vectorized alternative exists


## 8. Basic EDA Workflow - Putting It All Together

A typical exploratory data analysis (EDA) workflow follows this general order:

1. Load/inspect the data (`.head()`, `.info()`, `.shape`)
2. Check summary statistics (`.describe()`)
3. Check for missing values (`.isnull().sum()`)
4. Check for outliers (compare mean vs median, or use boxplot-style thresholds)
5. Handle missing values and outliers appropriately
6. Explore relationships (`.groupby()`, correlations)
7. Engineer/derive new features if needed


In [18]:
# Step 1 & 2: Inspect and summarize
print("Shape:", df.shape)
print("\nSummary statistics:")
print(df.describe())


Shape: (10, 12)

Summary statistics:
       customer_id       age  annual_income  account_balance  tenure_years  \
count     10.00000  10.00000       8.000000         10.00000      10.00000   
mean       5.50000  37.80000     168.312500         52.71000       6.00000   
std        3.02765  11.91451     316.500839         65.60514       6.63325   
min        1.00000  23.00000      30.000000         -2.50000       0.00000   
25%        3.25000  29.50000      43.375000          5.75000       1.25000   
50%        5.50000  36.00000      58.500000         28.85000       3.50000   
75%        7.75000  44.00000      77.000000         74.17500       7.50000   
max       10.00000  60.00000     950.000000        200.00000      20.00000   

       num_products  is_active         FD  
count     10.000000  10.000000  10.000000  
mean       2.200000   0.700000   0.500000  
std        1.398412   0.483046   0.527046  
min        1.000000   0.000000   0.000000  
25%        1.000000   0.250000   0.00000

In [19]:
# Step 3: Missing values check
print("Missing values:\n", df.isnull().sum())


Missing values:
 customer_id                0
age                        0
gender                     0
city                       0
annual_income              2
account_balance            0
tenure_years               0
num_products               0
is_active                  0
FD                         0
balance_flag_vectorized    0
balance_flag_apply         0
dtype: int64


In [20]:
# Step 4: Outlier check - compare mean vs median for annual_income
mean_income = df["annual_income"].mean()
median_income = df["annual_income"].median()
print(f"Mean income: {mean_income:.2f}")
print(f"Median income: {median_income:.2f}")
print(f"Difference suggests a possible outlier: {'YES' if abs(mean_income - median_income) > 50 else 'NO'}")

# A simple outlier flag using a threshold (3x the median, just for demonstration)
outlier_customers = df[df["annual_income"] > 3 * median_income]
print("\nPossible outlier customers:")
print(outlier_customers[["customer_id", "annual_income"]])


Mean income: 168.31
Median income: 58.50
Difference suggests a possible outlier: YES

Possible outlier customers:
   customer_id  annual_income
8            9          950.0


**Line by line explanation:**
- Comparing `mean` vs `median` is a simple, fast way to sense-check for outliers - a big gap between the two (as we see here) is a red flag
- `df["annual_income"] > 3 * median_income` -> a simple heuristic boolean mask to flag unusually large values (in real projects, more rigorous methods like IQR or z-score are used)
- This correctly isolates customer 9, our deliberately planted outlier


In [21]:
# Step 5: Handle missing values and outliers (using median, which we already established is more robust)
df_clean = df.copy()
df_clean["annual_income"] = df_clean["annual_income"].fillna(median_income)

# Step 6: Explore relationships - correlation between numeric features and FD
numeric_cols = ["age", "annual_income", "account_balance", "tenure_years", "num_products", "is_active", "FD"]
correlation_with_fd = df_clean[numeric_cols].corr()["FD"].sort_values(ascending=False)
print("Correlation of each feature with FD (target):")
print(correlation_with_fd)


Correlation of each feature with FD (target):
FD                 1.000000
account_balance    0.772994
age                0.760853
num_products       0.753778
tenure_years       0.699206
is_active          0.654654
annual_income      0.376087
Name: FD, dtype: float64


**Line by line explanation:**
- `.fillna(median_income)` -> we finalize our cleaning decision from Section 4, using median imputation
- `.corr()` -> computes pairwise **Pearson correlation** between numeric columns; values close to +1 mean strong positive relationship, close to -1 mean strong negative relationship, close to 0 mean little/no linear relationship
- `["FD"]` -> extracts just the correlation of every feature specifically with our target variable `FD`
- `tenure_years`, `account_balance`, and `num_products` show strong positive correlation with `FD` - customers who stay longer, hold more products, and have higher balances are more likely to have an FD - a genuinely useful EDA insight


In [22]:
# Step 7: A simple derived feature - income per tenure year (avoiding division by zero)
df_clean["income_per_tenure_year"] = df_clean["annual_income"] / df_clean["tenure_years"].replace(0, np.nan)
print(df_clean[["customer_id", "annual_income", "tenure_years", "income_per_tenure_year"]])


   customer_id  annual_income  tenure_years  income_per_tenure_year
0            1           45.0             1               45.000000
1            2           62.0             3               20.666667
2            3           58.5             8                7.312500
3            4           38.5             1               38.500000
4            5           95.0            15                6.333333
5            6           30.0             0                     NaN
6            7           71.0             6               11.833333
7            8           55.0             4               13.750000
8            9          950.0            20               47.500000
9           10           58.5             2               29.250000


**Line by line explanation:**
- `.replace(0, np.nan)` -> temporarily replaces `0` values in `tenure_years` with `NaN` before dividing, to avoid a **division by zero error** (customer 6 has `tenure_years = 0`)
- This is a simple example of **feature engineering** - creating a new, potentially useful column derived from existing ones
- Notice customer 6 correctly shows `NaN` in the new column instead of crashing the program or showing `inf`


## 9. Common Mistakes (Across Day 4 Topics)

- Confusing `.loc` (label-based) with `.iloc` (position-based), especially when the DataFrame's index isn't the default 0,1,2...
- Forgetting `df.copy()` before modifying a DataFrame - modifying a slice of the original can trigger the infamous `SettingWithCopyWarning`
- Using `.fillna(mean)` blindly without checking for outliers first (mean gets skewed, median is often safer)
- Using `how="inner"` merges without realizing rows silently disappear when there's no match
- Overusing `.apply()` with custom Python functions when a vectorized alternative (`np.where`, direct column arithmetic) exists - major performance cost on large data
- Dividing by a column that may contain zero without handling it first -> causes `inf` or errors


## 10. Best Practices (Across Day 4 Topics)

- Always run `.info()` and `.describe()` immediately after loading any new dataset
- Always check `.isnull().sum()` before doing any aggregation or modeling
- Prefer `median` over `mean` for imputation when outliers are present (verify using `.describe()` first)
- Use `.loc` for label/condition-based access, `.iloc` for pure positional access - don't mix them up
- Choose the correct join type (`how=`) deliberately - default to `"left"` when you want to preserve all rows of your primary dataset
- Prefer vectorized operations (`np.where`, direct arithmetic) over `.apply()` whenever possible
- Always use `.copy()` when you intend to modify a DataFrame without affecting the original


## 11. Practice Problems (No Solutions)

**Easy**
1. Using `.loc`, select all customers who are `is_active == 1`
2. Using `.iloc`, select the last 3 rows of the DataFrame
3. Find how many missing values exist in the entire DataFrame (all columns combined)

**Medium**
4. Group the data by `gender` and find the average `annual_income` (use median instead of mean, and explain why in a comment)
5. Merge `df` with a new small DataFrame containing a `risk_category` per `city`, using a `left` join, and check which rows have missing `risk_category`
6. Create a new column `is_senior` that is `True` if `age > 40`, using a vectorized approach (not `.apply()`)

**Hard**
7. Write a `.groupby("FD").agg(...)` call that computes the mean, min, and max of `account_balance` for each FD group in a single call
8. Using `.apply()` with a custom function, create a new column `income_level` with values `"low"` (income < 50), `"medium"` (50-100), `"high"` (>100) - handle the NaN values gracefully inside your function
9. Compare the outcome of an `inner` join vs `outer` join between `df` and `branch_info` on `city` - explain (in a markdown cell) exactly which rows differ and why


## 12. Revision Summary

- **Series** = single labeled column; **DataFrame** = full table of Series sharing an index
- Pandas allows **mixed dtypes per column** in one table - solves the limitation we hit with NumPy in Day 3
- `.head()`, `.info()`, `.describe()`, `.shape`, `.dtypes` -> first steps for exploring any new dataset
- `.loc` = label-based indexing; `.iloc` = position-based indexing
- Missing data: `.isnull()` to detect, `.dropna()` to remove, `.fillna()` to impute (prefer median over mean when outliers exist)
- `.groupby()` follows split -> apply -> combine; use `.agg()` for multiple aggregations at once
- `pd.merge()` types: inner (matches only), left (keep all of left), right (keep all of right), outer (keep everything)
- **Vectorized operations** (`np.where`, direct column math) are much faster than `.apply()` with custom functions - use `.apply()` only when logic can't be vectorized
- A basic EDA workflow: inspect -> summarize -> check missing values -> check outliers -> clean -> explore relationships (correlation, groupby) -> engineer features

> Interview tip: Be ready to explain, using this exact dataset as an example: why we used median over mean for `annual_income`, the difference between `.loc`/`.iloc`, and why vectorized operations beat `.apply()`.

---
*Day 4 (Pandas Deep Dive) complete in this single end-to-end notebook. Next: return to Day 3 to finish the remaining NumPy notebooks (indexing/slicing, boolean masking, broadcasting, vectorization vs loops, NumPy in EDA workflows).*
